In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [3]:
model_name = "roberta-base"
cache_dir = "./hf_cache"

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=5, 
    cache_dir=cache_dir
)

print("downloaded")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


downloaded


In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer

train_df = pd.read_csv("data/train.csv")
eval_df = pd.read_csv("data/eval.csv")

tokenizer = AutoTokenizer.from_pretrained("roberta-base", cache_dir="./hf_cache")

label2id = {name: i for i, name in enumerate(sorted(train_df["artist"].unique()))}
id2label = {i: name for name, i in label2id.items()}
print(label2id)

{'Death': 0, 'Gojira': 1, 'Meshuggah': 2, 'Opeth': 3, 'Tool': 4}


In [ ]:
from torch.utils.data import Dataset

class LyricsDataset(Dataset):
    def __init__(self, df, tokenizer, label2id, max_length=512):
        self.encodings = tokenizer(
            df["clean"].tolist(), 
            truncation=True,   
            padding=True,
            max_length=max_length, 
            return_tensors="pt"
        )
        self.labels = torch.tensor([label2id[a] for a in df["artist"]])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = LyricsDataset(train_df, tokenizer, label2id)
eval_dataset = LyricsDataset(eval_df, tokenizer, label2id)
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

Train: 459, Eval: 51


In [6]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base", 
    num_labels=5, 
    label2id=label2id, 
    id2label=id2label,
    cache_dir="./hf_cache"
)

args = TrainingArguments(
    output_dir="./classifier_output",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=10,
    weight_decay=0.01,
    learning_rate=2e-5,
    warmup_ratio=0.1,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": (preds == labels).mean()}

trainer = Trainer(
    model=model, 
    args=args, 
    train_dataset=train_dataset,
    eval_dataset=eval_dataset, 
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/Users/aliozkaya/uni/467/term_project/src/.venv/lib/python3.13/site-packages/torch/utils/data/dataloa

Epoch,Training Loss,Validation Loss,Accuracy
1,1.261016,0.952847,0.705882
2,0.543941,0.476524,0.803922
3,0.228934,0.427971,0.843137
4,0.121621,0.389617,0.921569
5,0.011556,0.405523,0.921569


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/aliozkaya/uni/467/term_project/src/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

/Users/aliozkaya/uni/467/term_project/src/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/aliozkaya/uni/467/term_project/src/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=290, training_loss=0.4880716973080717, metrics={'train_runtime': 303.0655, 'train_samples_per_second': 7.573, 'train_steps_per_second': 0.957, 'total_flos': 603856136954880.0, 'train_loss': 0.4880716973080717, 'epoch': 5.0})

In [7]:
from sklearn.metrics import classification_report, confusion_matrix

preds = trainer.predict(eval_dataset)
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = preds.label_ids

print(classification_report(
    y_true, y_pred, 
    target_names=[id2label[i] for i in range(5)]
))
print(confusion_matrix(y_true, y_pred))

/Users/aliozkaya/uni/467/term_project/src/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

       Death       1.00      1.00      1.00        12
      Gojira       0.75      0.75      0.75         8
   Meshuggah       1.00      0.92      0.96        12
       Opeth       0.92      1.00      0.96        12
        Tool       0.86      0.86      0.86         7

    accuracy                           0.92        51
   macro avg       0.91      0.90      0.90        51
weighted avg       0.92      0.92      0.92        51

[[12  0  0  0  0]
 [ 0  6  0  1  1]
 [ 0  1 11  0  0]
 [ 0  0  0 12  0]
 [ 0  1  0  0  6]]


In [10]:
trainer.save_model("./classifier_output/best_model")
tokenizer.save_pretrained("./classifier_output/best_model")
print("saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved


In [9]:
all_texts = pd.concat([train_df, eval_df])["clean"].tolist()
lengths = [len(tokenizer(text, truncation=False)["input_ids"]) for text in all_texts]
import numpy as np
print(f"Min: {np.min(lengths)}")
print(f"Median: {np.median(lengths)}")
print(f"Mean: {np.mean(lengths):.0f}")
print(f"Max: {np.max(lengths)}")
print(f"Songs > 512 tokens: {sum(1 for l in lengths if l > 512)} / {len(lengths)}")
print(f"Songs > 1024 tokens: {sum(1 for l in lengths if l > 1024)} / {len(lengths)}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (578 > 512). Running this sequence through the model will result in indexing errors


Min: 31
Median: 216.5
Mean: 238
Max: 968
Songs > 512 tokens: 17 / 510
Songs > 1024 tokens: 0 / 510
